# 13 — Gold DLT: Dimensions
dim_date (static calendar), dim_street (SCD2), dim_node_location (SCD2).

In [0]:
import dlt
from pyspark.sql import functions as F

CATALOG = spark.conf.get("pipeline.catalog", "vstone_catalog")
SILVER = f"{CATALOG}.{spark.conf.get('pipeline.silver_schema', 'silver')}"

GOLD_PROPS = {
    "quality": "gold",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}

##dim_date — static calendar, no SCD (a calendar doesn't change)

In [0]:
@dlt.table(
    name="dim_date",
    comment="Gold Static: gap-free calendar 2023-01-01 to 2025-12-31 (covers the real data "
            "window 2023-06-02 to 2024-03-11 plus buffer). PK: date_key. "
            "FK target: fact_street_readings.reading_date, fact_traffic_counts.reading_date, "
            "fact_citizen_reports.message_date -> dim_date.date_key.",
    table_properties={**GOLD_PROPS, "type": "static", "pk": "date_key"},
)
def dim_date():
    return (
        spark.range(1)
        .selectExpr("explode(sequence(to_date('2023-01-01'), to_date('2025-12-31'), interval 1 day)) as date_key")
        .select(
            F.col("date_key"),
            F.year("date_key").alias("year"),
            F.quarter("date_key").alias("quarter"),
            F.month("date_key").alias("month"),
            F.date_format("date_key", "MMMM").alias("month_name"),
            F.weekofyear("date_key").alias("week_of_year"),
            F.dayofmonth("date_key").alias("day"),
            F.date_format("date_key", "EEEE").alias("day_name"),
            F.when(F.dayofweek("date_key").isin(1, 7), True).otherwise(False).alias("is_weekend"),
            F.current_timestamp().alias("gold_load_dt"),
        )
    )

## dim_street — SCD2

In [0]:
@dlt.view(name="dim_street_source")
def dim_street_source():
    """MUST use readStream — apply_changes() requires a streaming source."""
    return spark.readStream.format("delta").table(f"{SILVER}.streets_list_silver")


dlt.create_streaming_table(
    name="dim_street",
    comment="Gold SCD2: street master data from streets_list_silver. "
            "Natural key = surrogate key: street_id INT (already a clean small int, "
            "no crc32 hash needed unlike the reference project's composite-string keys). "
            "FK target: fact_street_readings.street_id, fact_citizen_reports.street_id -> dim_street.street_id. "
            "SCD2: __START_AT / __END_AT. Active rows: __END_AT IS NULL.",
    table_properties={**GOLD_PROPS, "scd_type": "2", "pk": "street_id"},
)
dlt.apply_changes(
    target="dim_street",
    source="dim_street_source",
    keys=["street_id"],
    sequence_by="silver_load_dt",
    stored_as_scd_type=2,
    except_column_list=["bronze_load_dt", "bronze_source"],
)


## dim_node_location — SCD2

In [0]:
@dlt.view(name="dim_node_location_source")
def dim_node_location_source():
    return spark.readStream.format("delta").table(f"{SILVER}.node_locations_silver")


dlt.create_streaming_table(
    name="dim_node_location",
    comment="Gold SCD2: traffic sensor node coordinates from node_locations_silver "
            "(the (0,0)-coordinate row for location=7 was already quarantined in Silver — "
            "not present here). Natural key = surrogate key: location INT. "
            "FK target: fact_traffic_counts.location -> dim_node_location.location. "
            "SCD2: __START_AT / __END_AT. Active rows: __END_AT IS NULL.",
    table_properties={**GOLD_PROPS, "scd_type": "2", "pk": "location"},
)
dlt.apply_changes(
    target="dim_node_location",
    source="dim_node_location_source",
    keys=["location"],
    sequence_by="silver_load_dt",
    stored_as_scd_type=2,
    except_column_list=["bronze_load_dt", "bronze_source"],
)